# Сырые оси магнитометров в зависимости от силы сжатия протеза

Данные: `tools/ramp_*.csv` из `force_ramp.py` — несколько циклов, сила 10 → 100 % с шагом 1 %.
Колонки: `t_s, force_pct, THUMB_MLX0_X … MIDDLE_MLX3_Z` (сырые int16, LSB).

**Метод.** Каждый цикл делится на ступени — подряд идущие кадры с одной и той же силой (~0,5 с, ~30 кадров).
Для каждой ступени и каждой оси берётся **медиана** — она отбрасывает выбросы на переходе между ступенями.
Получается одна точка на (цикл, сила) → дискретные графики.
Ступень контакта (минимальная сила, ~2 с, в начале пальцы ещё едут к предмету) — медиана только последних `CONTACT_TAIL_S` секунд.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# --- параметры
FILE = Path('../tools/ramp_20261026_170237.csv')   # путь относительно папки ноутбука
CONTACT_TAIL_S = 0.5     # для ступени контакта: медиана последних N секунд
SAVE_PNG = True          # сохранять графики в analysis/
CELLS = ['THUMB', 'INDEX', 'MIDDLE']
AXES = ['X', 'Y', 'Z']
N_MLX = 4

In [ ]:
# --- загрузка
with open(FILE, encoding='utf-8-sig') as fh:
    rows = list(csv.reader(fh))
header, data = rows[0], np.array([[float(v) if v != '' else np.nan for v in r] for r in rows[1:]])
t, force = data[:, 0], data[:, 1]
mag = data[:, 2:]
mag_names = header[2:]
col = {n: i for i, n in enumerate(mag_names)}

# --- циклы: начало — переход force 0 -> >0
grip = force > 0
starts = np.where(grip & ~np.r_[False, grip[:-1]])[0]
cycle = np.zeros(len(t), int)              # 0 — вне хвата (раскрытие)
for k, s in enumerate(starts, 1):
    end = starts[k] if k < len(starts) else len(t)
    cycle[s:end] = k
cycle[~grip] = 0
n_cycles = len(starts)
print(f'{FILE.name}: {len(t)} кадров, {t[-1]:.1f} с, {len(t)/t[-1]:.1f} кадр/с, циклов: {n_cycles}')

In [ ]:
# --- медиана по каждой ступени (цикл, сила)
levels = np.unique(force[grip]).astype(int)
f_min = levels.min()
med = np.full((n_cycles, len(levels), mag.shape[1]), np.nan)   # [цикл, уровень силы, ось]
n_frames = np.zeros((n_cycles, len(levels)), int)

for c in range(1, n_cycles + 1):
    for j, L in enumerate(levels):
        sel = np.where((cycle == c) & (force == L))[0]
        if len(sel) == 0:
            continue
        if L == f_min:                                     # ступень контакта — только хвост
            sel = sel[t[sel] >= t[sel[-1]] - CONTACT_TAIL_S]
        med[c - 1, j] = np.median(mag[sel], axis=0)
        n_frames[c - 1, j] = len(sel)

print('кадров на ступень (медиана по всем):', int(np.median(n_frames[n_frames > 0])),
      '| мин:', n_frames[n_frames > 0].min(), '| макс:', n_frames.max())
med_all = np.nanmedian(med, axis=0)                        # медиана по циклам

In [ ]:
# --- графики: фигура на ячейку, сетка MLX (строки) × ось (столбцы), точки — циклы
colors = plt.cm.viridis(np.linspace(0, 0.9, n_cycles))

for cell in CELLS:
    fig, axs = plt.subplots(N_MLX, len(AXES), figsize=(15, 13), sharex=True)
    fig.suptitle(f'{cell}: медиана оси на каждой ступени силы ({n_cycles} циклов)', fontsize=14)
    for m in range(N_MLX):
        for a, axis in enumerate(AXES):
            ax = axs[m, a]
            i = col[f'{cell}_MLX{m}_{axis}']
            for c in range(n_cycles):
                ax.plot(levels, med[c, :, i], 'o', ms=3, color=colors[c], label=f'цикл {c + 1}')
            ax.plot(levels, med_all[:, i], '-', color='black', lw=1, label='медиана по циклам')
            ax.set_title(f'{cell}_MLX{m}_{axis}', fontsize=10)
            ax.grid(alpha=0.3)
            ax.tick_params(labelsize=8)
            if m == N_MLX - 1:
                ax.set_xlabel('сила сжатия, %')
            if a == 0:
                ax.set_ylabel('LSB')
    axs[0, 0].legend(fontsize=7, loc='best')
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    if SAVE_PNG:
        fig.savefig(f'{FILE.stem}_{cell}.png', dpi=110)
    plt.show()

In [ ]:
# --- таблица медиан: cycle, force_pct, n_frames, все 36 осей (для дальнейшего анализа)
out = Path(f'{FILE.stem}_medians.csv')
with open(out, 'w', newline='', encoding='utf-8-sig') as fh:
    w = csv.writer(fh)
    w.writerow(['cycle', 'force_pct', 'n_frames'] + mag_names)
    for c in range(n_cycles):
        for j, L in enumerate(levels):
            if n_frames[c, j]:
                w.writerow([c + 1, L, n_frames[c, j]] + [f'{v:.1f}' for v in med[c, j]])
print('сохранено:', out.resolve())